In [6]:
import os
import numpy as np
import pandas as pd

import allel

from mpire import WorkerPool

VCF_PATH = "/share/home/zhoujb/project/riceDataComb/processData/1-0-4-Soybean_2795/rawData/regionSNP/"
HAPDF_PATH = "/share/home/zhoujb/project/riceDataComb/processData/1-0-4-Soybean_2795/rawData/2-allGeneHapDf/"
HAPPER_PATH = "/share/home/zhoujb/project/riceDataComb/processData/1-0-4-Soybean_2795/rawData/2-allGeneHapPer/"

In [ ]:
def allelVCF2HapPer(gene_id):
    # 读取VCF文件
    vcf_file = os.path.join(VCF_PATH, "Chr{}.vcf".format(gene_id))
    
    if not os.path.exists(vcf_file):
        print("{} not exist".format(vcf_file))
        return
        
    callset = allel.read_vcf(vcf_file)

    samples = callset['samples']
    samples = [x.split("_")[0] for x in samples]

    gt = allel.GenotypeArray(callset['calldata/GT'])
    chrom = callset['variants/CHROM']
    pos = callset['variants/POS']

    # 将 GT（二倍体）转换为 “1维基因型编码”
    # 0→ref/ref, 1→杂合, 2→alt/alt
    geno = gt.to_n_alt()   # shape = (num_snps, num_samples)
    geno = geno.T                # 转换成样本 × SNP 矩阵

    geno_df = pd.DataFrame(geno, columns=pos, index=samples)
    geno_df.index.name = "sample"

    # 将每个样本的 SNP 型组合成 tuple，这样可作为 dict key
    hap_strings = [tuple(row) for row in geno]
    
    hap_df = pd.DataFrame({'sample': samples,'haplotype': hap_strings})

    # 确定 unique haplotypes，并编号
    unique_haps = hap_df['haplotype'].unique()
    hap_to_id = {h: f"Hap{idx+1}" for idx, h in enumerate(unique_haps)}

    hap_df['haplotype_id'] = hap_df['haplotype'].map(hap_to_id)

    freq = hap_df.groupby(['haplotype_id']).size().reset_index(name='count')
    freq['percentage'] = freq['count'] / freq['count'].sum()
    freq_d = freq.set_index('haplotype_id').to_dict()
    freq2count = freq_d["count"]
    freq2per = freq_d["percentage"]

    hap_df["count"] = hap_df['haplotype_id'].map(freq2count)
    hap_df["percentage"] = hap_df['haplotype_id'].map(freq2per)

    sample2hap = hap_df[["sample", "haplotype_id"]].set_index("sample").to_dict()["haplotype_id"]
    geno_df["haplotype_id"] = geno_df.index.map(sample2hap)

    geno_df.to_csv(os.path.join(HAPDF_PATH, "{}.txt".format(gene_id)), sep="\t")
    hap_df[['sample', 'haplotype_id', 'count', 'percentage']].to_csv(os.path.join(HAPPER_PATH, "{}.txt".format(gene_id)), sep="\t", index=0)

    print("{} DONE".format(gene_id))

    return


if __name__ == "__main__":
    in_file_path = "/share/home/zhoujb/project/riceDataComb/processData/1-0-4-Soybean_2795/rawData/soybean-region.txt"
    info_file = pd.read_table(in_file_path)
    
    all_gene_list = (
    info_file[["Chr", "Start", "End"]]
    .astype(str)
    .agg("_".join, axis=1)
    .tolist()
    )

    with WorkerPool(n_jobs=16) as pool:
        pool.map(allelVCF2HapPer, all_gene_list, progress_bar=True)

    print("ALL DONE")

Exception occurred, terminating ... :   0%|                                                                                                                                         | 0/91 [01:06<?, ?it/s]
